# GUMELAB · Sentimiento y emojis
Cuaderno para la sección 6.6 de la tesis. Calcula el sentimiento de **502.619 textos** (comentarios y respuestas de YouTube, tuits de X) con el modelo **RoBERTuito** de `pysentimiento`, **después de quitar los emojis** del texto, y guarda aparte los emojis de cada texto.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`. Luego `Entorno de ejecución → Ejecutar todas`.

El cuaderno guarda el avance en tu Google Drive por bloques: si Colab se desconecta, vuelve a ejecutar todo y continúa donde iba.

**Al final** descarga de la carpeta `MyDrive/GUMELAB_sentimiento/` estos dos archivos y cópialos a `datos_privados/derivados/`:
- `sentimiento_gumelab.csv.gz`
- `Emoji_Sentiment_Data_v1.0.csv`

In [ ]:
# 1. Verificar GPU e instalar librerías
!nvidia-smi -L
!pip -q install pysentimiento emoji

In [ ]:
# 2. Montar Drive y ubicar los datos
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil
CARPETA = '/content/drive/MyDrive/GUMELAB_sentimiento'
os.makedirs(CARPETA + '/bloques', exist_ok=True)
cand = [f for f in glob.glob(CARPETA + '/textos_gumelab*') if f.endswith(('.zip', '.csv.gz', '.csv'))]
if not cand:
    print('No encontré textos_gumelab en Drive; súbelo ahora (se abrirá el selector de archivos)...')
    from google.colab import files
    up = files.upload()
    nombre = list(up)[0]; shutil.move(nombre, CARPETA + '/' + nombre)
    cand = [CARPETA + '/' + nombre]
ENTRADA = cand[0]
print('Entrada:', ENTRADA, round(os.path.getsize(ENTRADA)/1e6,1), 'MB')

In [ ]:
# 3. Leer textos, extraer emojis y quitar emojis del texto
import pandas as pd, emoji, re
d = pd.read_csv(ENTRADA, dtype={'id': str})
MOD = re.compile('[\U0001F3FB-\U0001F3FF\uFE0F\u200d]')   # tonos de piel, selector de variación, ZWJ
def emojis_de(t):
    # emoji.emoji_list respeta secuencias compuestas (familias, banderas)
    return ' '.join(MOD.sub('', e['emoji']) or e['emoji'] for e in emoji.emoji_list(t))
d['texto'] = d['texto'].astype(str)
d['emojis'] = d['texto'].map(emojis_de)
d['n_emojis'] = d['emojis'].str.split().str.len().fillna(0).astype(int)
d['texto_sin_emoji'] = d['texto'].map(lambda t: re.sub(r'\s+', ' ', emoji.replace_emoji(t, '')).strip())
d['solo_emoji'] = d['texto_sin_emoji'].str.replace(r'[\W_]+', '', regex=True).eq('')
print(len(d), 'textos |', (d.n_emojis>0).sum(), 'con emojis |', d.solo_emoji.sum(), 'solo emojis (sin texto para predecir)')

In [ ]:
# 4. Cargar el modelo RoBERTuito de sentimiento (pysentimiento)
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pysentimiento.preprocessing import preprocess_tweet
MODELO = 'pysentimiento/robertuito-sentiment-analysis'
tok = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForSequenceClassification.from_pretrained(MODELO).cuda().eval().half()
ETIQ = model.config.id2label   # {0:'NEG', 1:'NEU', 2:'POS'}
print(ETIQ)
@torch.no_grad()
def predecir(textos, lote=256):
    out = []
    for i in range(0, len(textos), lote):
        x = [preprocess_tweet(t, lang='es') for t in textos[i:i+lote]]
        enc = tok(x, padding=True, truncation=True, max_length=128, return_tensors='pt').to('cuda')
        out.append(torch.softmax(model(**enc).logits.float(), -1).cpu().numpy())
    return np.vstack(out)
print(predecir(['Qué buena serie, la mejor de Caracol', 'Esa serie es una apología al narcotráfico']).round(3))

In [ ]:
# 5. Predicción por bloques (guarda avance en Drive)
import time
p = d[~d.solo_emoji].copy()
p['largo'] = p.texto_sin_emoji.str.len()
p = p.sort_values('largo')            # lotes de longitud parecida = más rápido
BLOQUE = 25000
nb = (len(p) + BLOQUE - 1) // BLOQUE
t0 = time.time()
for b in range(nb):
    f = f'{CARPETA}/bloques/b{b:03d}.csv.gz'
    if os.path.exists(f): continue
    q = p.iloc[b*BLOQUE:(b+1)*BLOQUE]
    pr = predecir(q.texto_sin_emoji.tolist())
    r = pd.DataFrame({'fuente': q.fuente.values, 'id': q.id.values})
    for k, lab in ETIQ.items(): r['p_' + lab.lower()] = pr[:, k].round(4)
    r.to_csv(f, index=False, compression='gzip')
    print(f'bloque {b+1}/{nb}  {(time.time()-t0)/60:.1f} min')
print('Listo')

In [ ]:
# 6. Unir resultados y guardar
import glob
res = pd.concat([pd.read_csv(f, dtype={'id': str}) for f in sorted(glob.glob(CARPETA + '/bloques/*.csv.gz'))])
s = d.drop(columns=['texto']).merge(res, on=['fuente', 'id'], how='left')
cols = [c for c in s.columns if c.startswith('p_')]
s['pred'] = s[cols].idxmax(axis=1).str.replace('p_', '').str.upper()
s.loc[s.solo_emoji, 'pred'] = 'SOLO_EMOJI'
s = s.drop(columns=['texto_sin_emoji'])
s.to_csv(CARPETA + '/sentimiento_gumelab.csv.gz', index=False, compression='gzip')
print(s.pred.value_counts(normalize=True).round(3))
print(pd.crosstab([s.fuente, s.serie], s.pred, normalize='index').round(3))

In [ ]:
# 7. Descargar el Emoji Sentiment Ranking (Kralj Novak et al., 2015)
import urllib.request
URL = 'https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1048/Emoji_Sentiment_Data_v1.0.csv'
dest = CARPETA + '/Emoji_Sentiment_Data_v1.0.csv'
try:
    urllib.request.urlretrieve(URL, dest); print(pd.read_csv(dest).head())
except Exception as e:
    print('No se pudo descargar automáticamente:', e)
    print('Descárgalo a mano desde https://www.clarin.si/repository/xmlui/handle/11356/1048 y súbelo a Claude.')

In [ ]:
# 8. Descargar los archivos a tu computador (también quedan en Drive)
from google.colab import files
files.download(CARPETA + '/sentimiento_gumelab.csv.gz')
if os.path.exists(dest): files.download(dest)